# OSM Download via osmnx — FUA Polygon Batch

Downloads OSM layers for every city in the FUA shapefile using its polygon boundary.

**Layers saved per city (single GeoPackage):**
- `roads` — drive network
- `buildings` — footprints clipped to polygon
- `amenities` — all OSM amenity types
- `landuse` — land-use polygons
- `greenspace` — parks, forests, gardens

**Output structure:**
```
<OUTPUT_ROOT>/<city_name>/<city_name>_osm.gpkg
```

## 1. Imports

In [ ]:
from __future__ import annotations

import logging
import re
import traceback
from pathlib import Path
from typing import Literal, Sequence

import geopandas as gpd
import osmnx as ox
from shapely.geometry import MultiPolygon, Polygon

logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

## 2. Configuration

Edit the two paths below to match your environment.

In [ ]:
# Input shapefile
FUA_SHP  = r"D:\000_SCI\10_Compact_city\3_FUA_reference\GHS_FUA_cities_subset_clean.shp"
CITY_COL = "eFUA_name"   # column that holds the city name

# Root folder where per-city sub-folders will be created
OUTPUT_ROOT = r"D:\000_SCI\10_Compact_city\OSM_data"

# Road network type: "drive" | "all" | "walk" | "bike"
NETWORK_TYPE = "drive"

# Skip cities whose .gpkg already exists (resume-safe)
SKIP_EXISTING = True

NetworkType = Literal["all", "all_public", "bike", "drive", "drive_service", "walk"]

## 3. Helper Functions

In [ ]:
def load_fua_shapefile(
    shp_path: str | Path = FUA_SHP,
    city_col: str = CITY_COL,
) -> gpd.GeoDataFrame:
    """Load the FUA shapefile and reproject to EPSG:4326 if needed."""
    gdf = gpd.read_file(shp_path)
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)
    if city_col not in gdf.columns:
        raise KeyError(
            f"Column '{city_col}' not found. Available columns: {list(gdf.columns)}"
        )
    return gdf


def _safe_dirname(name: str) -> str:
    """Strip characters that are invalid in directory names."""
    return re.sub(r'[\\/:*?"<>|]', "_", name).strip()


def _keep_polygon_geom(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Keep only (Multi)Polygon geometries."""
    mask = gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    return gdf[mask].copy()


def _save_gdf(gdf: gpd.GeoDataFrame, path: str | Path, layer: str) -> None:
    """Save a GeoDataFrame to file; format inferred from extension."""
    path = Path(path)
    ext  = path.suffix.lower()
    if ext == ".gpkg":
        gdf.to_file(path, layer=layer, driver="GPKG")
    elif ext == ".shp":
        gdf.to_file(path)
    elif ext in {".geojson", ".json"}:
        gdf.to_file(path, driver="GeoJSON")
    else:
        gdf.to_file(path, layer=layer, driver="GPKG")

## 4. Download Functions

In [ ]:
def download_road_network(
    polygon: Polygon | MultiPolygon,
    network_type: NetworkType = "drive",
    output_path: str | Path | None = None,
) -> gpd.GeoDataFrame:
    """Download the road network inside *polygon*."""
    logger.info("  Downloading road network (network_type=%s) ...", network_type)
    G = ox.graph_from_polygon(polygon, network_type=network_type)
    _, edges = ox.graph_to_gdfs(G)
    edges = edges.reset_index()
    if output_path is not None:
        _save_gdf(edges, output_path, layer="roads")
    logger.info("    %d road segments.", len(edges))
    return edges

In [ ]:
def download_buildings(
    polygon: Polygon | MultiPolygon,
    output_path: str | Path | None = None,
) -> gpd.GeoDataFrame:
    """
    Download building footprints strictly inside *polygon*.

    - Only polygon geometries (footprints) are kept.
    - Results are clipped to the polygon boundary so no footprint
      extends outside it.
    """
    logger.info("  Downloading buildings ...")
    gdf = ox.features_from_polygon(polygon, tags={"building": True})
    gdf = _keep_polygon_geom(gdf)   # footprints only
    gdf = gdf.clip(polygon)          # clip to boundary
    if output_path is not None:
        _save_gdf(gdf, output_path, layer="buildings")
    logger.info("    %d building footprints.", len(gdf))
    return gdf

In [ ]:
def download_amenities(
    polygon: Polygon | MultiPolygon,
    amenity_filter: list[str] | None = None,
    output_path: str | Path | None = None,
) -> gpd.GeoDataFrame:
    """
    Download all OSM amenity features inside *polygon*.

    Keeps every geometry type (points, lines, polygons) because OSM
    amenities can be mapped as nodes, ways, or relations.
    Results are clipped to the polygon boundary.

    Parameters
    ----------
    amenity_filter : list of str, optional
        Restrict to specific amenity values, e.g. ["school", "hospital"].
        Pass None (default) to download ALL amenity types.
    """
    logger.info("  Downloading amenities ...")
    tags = {"amenity": amenity_filter if amenity_filter else True}
    gdf  = ox.features_from_polygon(polygon, tags=tags)
    gdf  = gdf.clip(polygon)
    if output_path is not None:
        _save_gdf(gdf, output_path, layer="amenities")
    logger.info("    %d amenity features.", len(gdf))
    return gdf

In [ ]:
def download_landuse(
    polygon: Polygon | MultiPolygon,
    output_path: str | Path | None = None,
) -> gpd.GeoDataFrame:
    """Download land-use polygons inside *polygon*."""
    logger.info("  Downloading land use ...")
    gdf = ox.features_from_polygon(polygon, tags={"landuse": True})
    gdf = _keep_polygon_geom(gdf)
    if output_path is not None:
        _save_gdf(gdf, output_path, layer="landuse")
    logger.info("    %d land-use polygons.", len(gdf))
    return gdf


def download_greenspace(
    polygon: Polygon | MultiPolygon,
    output_path: str | Path | None = None,
) -> gpd.GeoDataFrame:
    """Download parks, forests, and other green spaces inside *polygon*."""
    logger.info("  Downloading green spaces ...")
    tags = {
        "leisure": ["park", "garden", "nature_reserve", "recreation_ground"],
        "landuse": ["forest", "grass", "meadow", "orchard", "village_green"],
        "natural": ["wood", "scrub", "heath", "grassland"],
    }
    gdf = ox.features_from_polygon(polygon, tags=tags)
    gdf = _keep_polygon_geom(gdf)
    if output_path is not None:
        _save_gdf(gdf, output_path, layer="greenspace")
    logger.info("    %d green-space polygons.", len(gdf))
    return gdf

## 5. Batch Download — All Cities in FUA Shapefile

In [ ]:
def download_fua_batch(
    shp_path: str | Path = FUA_SHP,
    city_col: str = CITY_COL,
    output_root: str | Path = OUTPUT_ROOT,
    network_type: NetworkType = NETWORK_TYPE,
    skip_existing: bool = SKIP_EXISTING,
) -> None:
    """
    Iterate every row of the FUA shapefile and download OSM data
    for each city polygon.

    Output per city
    ---------------
    <output_root>/<city_name>/<city_name>_osm.gpkg
      layers: roads | buildings | amenities | landuse | greenspace
    """
    fua         = load_fua_shapefile(shp_path, city_col)
    output_root = Path(output_root)
    total       = len(fua)
    failed      = []

    logger.info("FUA shapefile loaded: %d cities to process.", total)

    for i, (idx, row) in enumerate(fua.iterrows(), start=1):
        city_name = str(row[city_col])
        safe_name = _safe_dirname(city_name)
        city_dir  = output_root / safe_name
        gpkg_path = city_dir / f"{safe_name}_osm.gpkg"

        logger.info("[%d/%d] %s", i, total, city_name)

        if skip_existing and gpkg_path.exists():
            logger.info("  Already exists — skipping.")
            continue

        polygon = row.geometry
        if polygon is None or polygon.is_empty:
            logger.warning("  Empty geometry — skipping.")
            continue

        city_dir.mkdir(parents=True, exist_ok=True)

        try:
            download_road_network(polygon, network_type, gpkg_path)
            download_buildings(polygon,                  gpkg_path)
            download_amenities(polygon, output_path=     gpkg_path)
            download_landuse(polygon,                    gpkg_path)
            download_greenspace(polygon,                 gpkg_path)
            logger.info("  Saved → %s", gpkg_path)
        except Exception:
            logger.error("  FAILED for %s:\n%s", city_name, traceback.format_exc())
            failed.append(city_name)

    logger.info("=" * 60)
    logger.info("Batch complete: %d / %d succeeded.", total - len(failed), total)
    if failed:
        logger.warning("Failed cities (%d):", len(failed))
        for name in failed:
            logger.warning("  - %s", name)

## 6. Run

### 6a. Batch — all cities in the shapefile

In [ ]:
download_fua_batch(
    shp_path     = FUA_SHP,
    city_col     = CITY_COL,
    output_root  = OUTPUT_ROOT,
    network_type = NETWORK_TYPE,
    skip_existing= SKIP_EXISTING,
)

### 6b. Single city (optional)

Uncomment and run this cell to download one specific city.

In [ ]:
# fua = load_fua_shapefile()
# row = fua[fua[CITY_COL] == "Seoul"].iloc[0]
# polygon = row.geometry

# city_name = "Seoul"
# city_dir  = Path(OUTPUT_ROOT) / city_name
# gpkg_path = city_dir / f"{city_name}_osm.gpkg"
# city_dir.mkdir(parents=True, exist_ok=True)

# download_road_network(polygon, NETWORK_TYPE, gpkg_path)
# download_buildings(polygon,                  gpkg_path)
# download_amenities(polygon, output_path=     gpkg_path)
# download_landuse(polygon,                    gpkg_path)
# download_greenspace(polygon,                 gpkg_path)
# print("Done:", gpkg_path)

### 6c. Preview downloaded data (optional)

In [ ]:
# import fiona
# gpkg = Path(OUTPUT_ROOT) / "Seoul" / "Seoul_osm.gpkg"
# print("Layers:", fiona.listlayers(str(gpkg)))

# roads     = gpd.read_file(gpkg, layer="roads")
# buildings = gpd.read_file(gpkg, layer="buildings")
# amenities = gpd.read_file(gpkg, layer="amenities")
# display(amenities[["amenity", "name", "geometry"]].head())